In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

from bell import *

In [2]:
d = 4
t = np.sqrt(2 + np.sqrt(5))
n = np.sqrt(5 + np.sqrt(5))

I = np.eye(d)
X = np.exp(1j*np.pi/4)*np.array([[0,1j,0,0],[-1,0,0,0],[0,0,0,1],[0,0,1j,0]])
Z = np.exp(1j*np.pi/4)*np.array([[0,0,-1,0],[0,0,0,1],[1j,0,0,0],[0,1j,0,0]])
D = lambda j1, j2: np.linalg.matrix_power(X, j1) @ np.linalg.matrix_power(Z, j2) 
U = np.array([[0,0,1,0],[0,0,0,1j],[1,0,0,0],[0,-1j,0,0]])
V = np.array([[0,1,0,0],[1,0,0,0],[0,0,0,-1j],[0,0,1j,0]])

phi = np.array([t, 1j, 1j, 1j])/n
phi_k = np.array([O @ phi for O in [I, U, V, U@V]])
csic = np.array([[D(j1, j2) @ phi for j2 in range(d) for j1 in range(d)] for phi in phi_k])
CSIC = np.array([[np.outer(psi, psi.conj())/d for psi in csic[i]] for i in range(4)])
VN = np.array([[np.outer(csic[j][i], csic[j][i].conj()) for j in range(4)] for i in range(16)])

In [3]:
d = 4
E = [[VN[2], VN[3], VN[4]], [VN[0], VN[1], CSIC[-1]]]
w = extract_specs(E)
n_parties = len(E)
deterministic_behaviors = construct_deterministic_behaviors(w)

ket = ghz(d, n_parties)
rho = np.outer(ket, ket.conj())
p = quantum_behavior_from_povms(E, rho)

In [4]:
L, problem = construct_hidden_variable_model(w, p, return_problem=True, deterministic_behaviors=deterministic_behaviors)
problem.status

'infeasible'

In [5]:
bell_functional, classical_bound, problem = construct_bell_inequality(w, p, return_problem=True, deterministic_behaviors=deterministic_behaviors)

print("LP objective:", problem.value)
print("p·s, S:", float(p @ bell_functional), float(classical_bound))

LP objective: 0.9999999871201908
p·s, S: 2246.702002127052 2245.7020021399317


In [8]:
reference_measurements = np.array([CSIC[0] for i in range(n_parties)])
reference_states = d*reference_measurements

T, T_meta = construct_T(rho, E, reference_measurements, reference_states, return_metadata=True)
phi = T_meta["phi"]
assert np.allclose(p, T @ phi)

T_singular_values = np.linalg.svd(T, compute_uv=False)
Delta = bell_functional @ p - classical_bound
bound = Delta/(np.max(T_singular_values)*np.linalg.norm(bell_functional)); bound

np.float64(0.0001291964342172171)